In [3]:
import pandas as pd
import numpy as np

# Load the CDC data - check all sheets first
excel_file = pd.ExcelFile('./data/raw/cdc_251644_DS1.xlsx')
print("Available sheets:")
for i, sheet in enumerate(excel_file.sheet_names):
    print(f"{i}: {sheet}")

FileNotFoundError: [Errno 2] No such file or directory: './data/raw/cdc_251644_DS1.xlsx'

In [ ]:
# Load state-level SIR comparison data 
# SIR > 1 means worse than national average, < 1 means better

# Read multiple state SIR sheets and combine
sir_data = []

for sheet_num in range(46, 53):  # Tables 10a through 10g
    sheet_name = excel_file.sheet_names[sheet_num]
    print(f"\nLoading {sheet_name}...")
    
    df = pd.read_excel('./data/raw/cdc_251644_DS1.xlsx', sheet_name=sheet_name, header=1)
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()[:5]}")  # First 5 columns
    print(df.head(3))

In [ ]:
# Column 0 = State names
# Column 1 = 2022 SIR 
# Column 2 = 2023 SIR 
# Column 3 = % Change
# Column 4 = Direction
# Column 5 = p value

# Rename columns to something readable
df_clabsi.columns = ['State', 'SIR_2022', 'SIR_2023', 'Percent_Change', 'Direction', 'P_Value']

print("New column names:")
print(df_clabsi.columns.tolist())
print("\nFirst 5 rows:")
print(df_clabsi.head())

In [ ]:
# fix row 0
df_clabsi = df_clabsi.iloc[1:]

print("After removing header row:")
print(df_clabsi.head())

In [ ]:
# convert SIR_2023 values from txt to number 
df_clabsi['SIR_2023'] = pd.to_numeric(df_clabsi['SIR_2023'], errors='coerce')

print("Data types:")
print(df_clabsi.dtypes)
print("\nFirst 5 rows:")
print(df_clabsi.head())
print("\nAny missing values?")
print(df_clabsi.isnull().sum())

In [ ]:
# I only need State and SIR_2023 for the dashboard
df_clabsi_clean = df_clabsi[['State', 'SIR_2023']].copy()

# Remove rows where SIR_2023 is missing
df_clabsi_clean = df_clabsi_clean.dropna(subset=['SIR_2023'])

# Rename the SIR_2023 column to include the infection name
df_clabsi_clean.columns = ['State', 'CLABSI_SIR_2023']

print("Clean CLABSI data:")
print(df_clabsi_clean.head(10))
print(f"\nShape: {df_clabsi_clean.shape}")

In [ ]:
# Define a function that cleans infection data

def extract_infection_data(sheet_name, infection_name):
    """
    Load and clean one infection type from CDC data.
    Returns: DataFrame with State and SIR_2023 columns
    """
    
    # Read the Excel sheet (skip first 2 rows of headers)
    df = pd.read_excel(
        './data/raw/cdc_251644_DS1.xlsx',
        sheet_name=sheet_name,
        skiprows=2
    )
    
    # Rename columns
    df.columns = ['State', 'SIR_2022', 'SIR_2023', 'Percent_Change', 'Direction', 'P_Value']
    
    # Remove header row (row 0)
    df = df.iloc[1:]
    
    # Convert SIR_2023 to numbers
    df['SIR_2023'] = pd.to_numeric(df['SIR_2023'], errors='coerce')
    
    # Keep only State and SIR_2023
    df_clean = df[['State', 'SIR_2023']].copy()
    
    # Remove missing values
    df_clean = df_clean.dropna(subset=['SIR_2023'])
    
    # Rename SIR_2023 to include infection name
    df_clean.columns = ['State', f'{infection_name}_SIR_2023']
    
    return df_clean

# Test the function on CAUTI
df_cauti = extract_infection_data('Table 10b-State SIR Comparison', 'CAUTI')
print("CAUTI data:")
print(df_cauti.head())

In [ ]:
infections = [
    ('Table 10a-State SIR Comparison', 'CLABSI'),
    ('Table 10b-State SIR Comparison', 'CAUTI'),
    ('Table 10c-State SIR Comparison ', 'VAE'),
    ('Table 10d-State SIR Comparison', 'SSI_COLON'),
    ('Table 10e-State SIR Comparison', 'SSI_HYST'),
    ('Table 10f-State SIR Comparison', 'MRSA'),
    ('Table 10g-State SIR Comparison', 'CDI'),
]
df_all = None

for sheet_name, infection_name in infections:
    print(f"Loading {infection_name}...")
    
    df_infection = extract_infection_data(sheet_name, infection_name)
    
    # Merge with the growing master dataframe
    if df_all is None:
        df_all = df_infection
    else:
        # Merge on State column 
        df_all = df_all.merge(df_infection, on='State', how='outer')
    
    print(f"  {infection_name}: {len(df_infection)} states")

print(f"\nFinal dataset shape: {df_all.shape}")
print(f"States: {len(df_all)}")
print(f"Columns: {df_all.columns.tolist()}")
print("\nFirst 10 rows:")
print(df_all.head(10))

In [5]:
import pandas as pd

# Load the cleaned data 
df_all = pd.read_csv('../data/processed/hai_state_data_2023.csv')

print(f"Data loaded: {df_all.shape}")
print(df_all.head())

Data loaded: (52, 8)
        State  CLABSI_SIR_2023  CAUTI_SIR_2023  VAE_SIR_2023  \
0     Alabama            0.919           0.629         1.025   
1      Alaska            0.512           0.909         1.814   
2     Arizona            0.677           0.405         0.797   
3    Arkansas            0.667           0.440         1.947   
4  California            0.751           0.715         1.182   

   SSI_COLON_SIR_2023  SSI_HYST_SIR_2023  MRSA_SIR_2023  CDI_SIR_2023  
0               0.732              0.970          1.022         0.473  
1               1.212              1.434          0.232         0.398  
2               0.825              1.266          0.719         0.463  
3               1.146              1.103          0.919         0.363  
4               0.871              0.818          0.713         0.492  


In [6]:
# What's the average SIR for each infection type nationally?
print("AVERAGE SIR BY INFECTION TYPE (2023):")

infections_cols = [
    'CLABSI_SIR_2023',
    'CAUTI_SIR_2023', 
    'VAE_SIR_2023',
    'SSI_COLON_SIR_2023',
    'SSI_HYST_SIR_2023',
    'MRSA_SIR_2023',
    'CDI_SIR_2023'
]
national_avg = df_all[infections_cols].mean()

# Sort from worst to best
national_avg_sorted = national_avg.sort_values(ascending=False)

print(national_avg_sorted)
print("INTERPRETATION:")
print("SIR < 1.0 = Better than national average")
print("SIR > 1.0 = Worse than national average")

AVERAGE SIR BY INFECTION TYPE (2023):
VAE_SIR_2023          1.203915
SSI_HYST_SIR_2023     1.105808
SSI_COLON_SIR_2023    0.904059
CLABSI_SIR_2023       0.738804
MRSA_SIR_2023         0.692608
CAUTI_SIR_2023        0.661353
CDI_SIR_2023          0.445294
dtype: float64
INTERPRETATION:
SIR < 1.0 = Better than national average
SIR > 1.0 = Worse than national average


In [7]:
# Which states are best at preventing CLABSI?
print("BEST 5 STATES FOR CLABSI:")
print(df_all.nsmallest(5, 'CLABSI_SIR_2023')[['State', 'CLABSI_SIR_2023']])

print("\nWORST 5 STATES FOR CLABSI:")
print(df_all.nlargest(5, 'CLABSI_SIR_2023')[['State', 'CLABSI_SIR_2023']])

print("\nBEST 5 STATES FOR VAE (biggest problem):")
print(df_all.nsmallest(5, 'VAE_SIR_2023')[['State', 'VAE_SIR_2023']])

print("\nWORST 5 STATES FOR VAE:")
print(df_all.nlargest(5, 'VAE_SIR_2023')[['State', 'VAE_SIR_2023']])

BEST 5 STATES FOR CLABSI:
           State  CLABSI_SIR_2023
26       Montana            0.347
12         Idaho            0.503
51       Wyoming            0.504
1         Alaska            0.512
34  North Dakota            0.523

WORST 5 STATES FOR CLABSI:
            State  CLABSI_SIR_2023
39    Puerto Rico            2.721
49  West Virginia            0.963
0         Alabama            0.919
11         Hawaii            0.891
19          Maine            0.874

BEST 5 STATES FOR VAE (biggest problem):
            State  VAE_SIR_2023
49  West Virginia         0.433
6     Connecticut         0.677
43      Tennessee         0.730
10        Georgia         0.760
2         Arizona         0.797

WORST 5 STATES FOR VAE:
         State  VAE_SIR_2023
12       Idaho         3.058
26     Montana         2.197
3     Arkansas         1.947
31  New Mexico         1.884
1       Alaska         1.814


In [9]:
# Create an "overall score" - average SIR across all 7 infections
df_all['Overall_Score'] = df_all[infections_cols].mean(axis=1)


print("BEST 10 STATES OVERALL (lowest average SIR):")
print(df_all.nsmallest(10, 'Overall_Score')[['State', 'Overall_Score']])

print("\nWORST 10 STATES OVERALL (highest average SIR):")
print(df_all.nlargest(10, 'Overall_Score')[['State', 'Overall_Score']])

print("\nNATIONAL AVERAGE OVERALL SCORE:")
print(f"{df_all['Overall_Score'].mean():.3f}")

print("\nSTANDARD DEVIATION:")
print(f"{df_all['Overall_Score'].std():.3f}")

BEST 10 STATES OVERALL (lowest average SIR):
             State  Overall_Score
6      Connecticut       0.622857
51         Wyoming       0.655000
8         Delaware       0.671833
40    Rhode Island       0.693714
42    South Dakota       0.698143
41  South Carolina       0.698286
45            Utah       0.698714
30      New Jersey       0.701000
47        Virginia       0.712857
43       Tennessee       0.720143

WORST 10 STATES OVERALL (highest average SIR):
           State  Overall_Score
39   Puerto Rico       1.168667
34  North Dakota       1.036333
11        Hawaii       1.010143
12         Idaho       0.996571
46       Vermont       0.943333
3       Arkansas       0.940714
22      Michigan       0.934143
1         Alaska       0.930143
24   Mississippi       0.917857
48    Washington       0.913000

NATIONAL AVERAGE OVERALL SCORE:
0.820

STANDARD DEVIATION:
0.105


In [10]:
# Add regions to the data
def assign_region(state):
    regions = {
        'Northeast': ['Connecticut', 'Maine', 'Massachusetts', 'New Hampshire', 'Rhode Island', 
                      'Vermont', 'New Jersey', 'New York', 'Pennsylvania'],
        'Midwest': ['Illinois', 'Indiana', 'Michigan', 'Ohio', 'Wisconsin', 'Iowa', 'Kansas', 
                    'Minnesota', 'Missouri', 'Nebraska', 'North Dakota', 'South Dakota'],
        'South': ['Delaware', 'Florida', 'Georgia', 'Maryland', 'North Carolina', 'South Carolina',
                  'Virginia', 'West Virginia', 'Alabama', 'Kentucky', 'Mississippi', 'Tennessee',
                  'Arkansas', 'Louisiana', 'Oklahoma', 'Texas'],
        'West': ['Arizona', 'Colorado', 'Idaho', 'Montana', 'Nevada', 'New Mexico', 'Utah', 
                 'Wyoming', 'Alaska', 'California', 'Hawaii', 'Oregon', 'Washington'],
    }
    for region, states in regions.items():
        if state in states:
            return region
    return 'Other'

df_all['Region'] = df_all['State'].apply(assign_region)

print("AVERAGE INFECTION RATE BY REGION:")
regional_avg = df_all.groupby('Region')['Overall_Score'].mean().sort_values()
print(regional_avg)

print("\nREGION PERFORMANCE:")
for region, score in regional_avg.items():
    if score < 0.820:
        status = "BETTER than national avg"
    else:
        status = "WORSE than national avg"
    print(f"{region:15} {score:.3f}  ({status})")

AVERAGE INFECTION RATE BY REGION:
Region
Northeast    0.784005
South        0.810124
Midwest      0.815730
West         0.830527
Other        1.010083
Name: Overall_Score, dtype: float64

REGION PERFORMANCE:
Northeast       0.784  (BETTER than national avg)
South           0.810  (BETTER than national avg)
Midwest         0.816  (BETTER than national avg)
West            0.831  (WORSE than national avg)
Other           1.010  (WORSE than national avg)
